In [1]:
import pandas as pd

In [ ]:

resultados = pd.read_csv("/home/aninha/Desktop/Doutorado/Modelos/pan_odontograma_2025_04_YOLOv11/metrics_all_images.csv")
resultados

,filename,class_name,true_pos,false_pos,false_neg,precision,recall,conf_thresh,iou_thresh,n_comp_label,n_comp_pred,label,label_name,pred,pred_name,scores
0,136735512_2_c000_panoramica,11,1,0,0,1.0,1.0,0.05,0.3,1,1,[[0.459872 0.306573 0.496594 0.570773]],['11'],[[0.45948529 0.32147181 0.4975225 0.56547451]],['11'],[0.84812677]
1,136735512_2_c000_panoramica,12,1,0,0,1.0,1.0,0.05,0.3,1,1,[[0.431405 0.3137855 0.467625 0.5504725]],['12'],[[0.42930353 0.32658541 0.46798173 0.54790121]],['12'],[0.80733562]
2,136735512_2_c000_panoramica,13,1,0,0,1.0,1.0,0.05,0.3,1,1,[[0.395533 0.2819945 0.439203 0.5475695]],['13'],[[0.39600271 0.29067567 0.43717235 0.54496706]],['13'],[0.83832932]
3,136735512_2_c000_panoramica,14,1,0,0,1.0,1.0,0.05,0.3,1,1,[[0.379014 0.326649 0.412162 0.524727]],['14'],[[0.37780908 0.32778287 0.41378468 0.5276143 ]],['14'],[0.70211136]
4,136735512_2_c000_panoramica,15,1,0,0,1.0,1.0,0.05,0.3,1,1,[[0.3487005 0.3213105 0.3879455 0.5302935]],['15'],[[0.34752119 0.32356015 0.389052 0.52942967]],['15'],[0.89612669]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7769,croppedpanoramica_142455886_6_c000_panoramica,44,1,0,0,1.0,1.0,0.35,0.3,1,1,[[0.3789325 0.7568905 0.4396295 0.8878355]],['44'],[[0.38231349 0.77250928 0.43182525 0.88934016]],['44'],[0.76139903]
7770,croppedpanoramica_142455886_6_c000_panoramica,45,1,0,0,1.0,1.0,0.05,0.3,1,1,[[0.3324195 0.771203 0.3899645 0.887417 ]],['45'],[[0.33290681 0.77509987 0.38201332 0.8808018 ]],['45'],[0.65331334]
7771,croppedpanoramica_142455886_6_c000_panoramica,46,1,0,0,1.0,1.0,0.05,0.3,1,1,[[0.2864255 0.6524205 0.3605225 0.8231235]],['46'],[[0.3362473 0.66762799 0.41068092 0.85800332]],['46'],[0.61611086]
7772,croppedpanoramica_142455886_6_c000_panoramica,47,1,0,0,1.0,1.0,0.65,0.3,1,1,[[0.234052 0.654256 0.302688 0.798116]],['47'],[[0.28524938 0.64709365 0.3641203 0.82662386]],['47'],[0.82951564]


In [5]:
len(resultados.filename.unique())

281

In [7]:
resultados = pd.read_csv("/home/aninha/Desktop/Doutorado/Modelos/pan_odontograma_2025_04_YOLOv11/metrics_summary.csv")
resultados

,true_pos,false_pos,false_neg,avg_accuracy,avg_precision,avg_recall,avg_f1,conf_thresh,iou_thresh,n_comp_label,n_comp_pred,n_img_label,n_img_pred
0,7373.0,240.0,937.0,0.862339,0.968475,0.887244,0.926082,0.16875,0.3,8310.0,7613.0,7549.0,7584.0


In [21]:
import pandas as pd
import math
import numpy as np

def gerar_tabela_sumario_latex(df, caption="Resultados do Modelo", label="tab:sumario"):
    remover = {"n_img_label", "n_img_pred", "conf_thresh", "iou_thresh"}
    df = df[[c for c in df.columns if c not in remover]].copy()

    rename_map = {
        "true_pos": "True Positive",
        "false_pos": "False Positive",
        "false_neg": "False Negative",
        "avg_accuracy": "Accuracy",
        "avg_precision": "Precision",
        "avg_recall": "Recall",
        "avg_f1": "F1-Score",
        "n_comp_label": "N Comp. Label",
        "n_comp_pred": "N Comp. Pred",
    }
    df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})

    def formatar_valor(x):
        if isinstance(x, (int, np.integer)):
            return str(x)
        if isinstance(x, (float, np.floating)):
            if float(x).is_integer():
                return str(int(x))
            return f"{x:.2f}"
        return str(x)

    # substitui applymap por apply(col.map)
    df_fmt = df.apply(lambda col: col.map(formatar_valor))

    cols = list(df_fmt.columns)
    n = len(cols)
    meio = math.ceil(n / 2)
    partes = [cols[:meio], cols[meio:]]

    tabelas_latex = []

    for subset in partes:
        subdf = df_fmt[subset]
        col_format = " ".join(["c"] * len(subset))

        latex_body = subdf.to_latex(
            index=False,
            escape=False,
            header=True,
            longtable=False,
            column_format=col_format
        )

        linhas = latex_body.split("\n")
        linhas_limpo = [linha for linha in linhas if
                        ("toprule" not in linha and
                         "midrule" not in linha and
                         "bottomrule" not in linha)]

        tabela_limpada = "\n".join(linhas_limpo)
        tabela_limpada = tabela_limpada.replace("\\\\", "\\\\ \\hline", 1)

        tabelas_latex.append(tabela_limpada)

    tabela_final = f"""
\\begin{{table}}[ht]
    \\centering
    \\caption{{{caption}}}
    \\label{{{label}}}

{tabelas_latex[0]}

\\vspace{{0.5cm}}

{tabelas_latex[1]}

\\end{{table}}
"""
    return tabela_final.strip()


In [22]:

# Formata com 3 casas decimais (ou ajuste como quiser)
resultados_fmt = resultados.round(3)

# Converte para LaTeX
latex = gerar_tabela_sumario_latex(resultados, caption="Overall Performance of the Tooth Detection Model Based on FDI Numbering", label="tab:sumario_odontograma")

print(latex)

\begin{table}[ht]
    \centering
    \caption{Overall Performance of the Tooth Detection Model Based on FDI Numbering}
    \label{tab:sumario_odontograma}

\begin{tabular}{c c c c c}
True Positive & False Positive & False Negative & Accuracy & Precision \\ \hline
7373 & 240 & 937 & 0.86 & 0.97 \\
\end{tabular}


\vspace{0.5cm}

\begin{tabular}{c c c c}
Recall & F1-Score & N Comp. Label & N Comp. Pred \\ \hline
0.89 & 0.93 & 8310 & 7613 \\
\end{tabular}


\end{table}


In [23]:
resultados_dente = pd.read_csv("/home/aninha/Desktop/Doutorado/Modelos/pan_odontograma_2025_04_YOLOv11/metrics_classes_summary.csv")
resultados_dente

,class_name,true_pos,false_pos,false_neg,avg_accuracy,avg_precision,avg_recall,avg_f1,conf_thresh,iou_thresh,n_comp_label,n_comp_pred,n_img_label,n_img_pred
0,11,261.0,7.0,31.0,0.872910,0.973881,0.893836,0.932143,0.05,0.3,292.0,268.0,264.0,268.0
1,12,250.0,11.0,31.0,0.856164,0.957854,0.889680,0.922509,0.05,0.3,281.0,261.0,253.0,260.0
2,13,258.0,7.0,34.0,0.862876,0.973585,0.883562,0.926391,0.05,0.3,292.0,265.0,264.0,265.0
3,14,231.0,9.0,33.0,0.846154,0.962500,0.875000,0.916667,0.05,0.3,264.0,240.0,239.0,240.0
4,15,224.0,13.0,31.0,0.835821,0.945148,0.878431,0.910569,0.05,0.3,255.0,237.0,231.0,236.0
5,16,223.0,3.0,32.0,0.864341,0.986726,0.874510,0.927235,0.05,0.3,255.0,226.0,233.0,225.0
6,17,229.0,9.0,35.0,0.838828,0.962185,0.867424,0.912351,0.05,0.3,264.0,238.0,239.0,238.0
7,18,133.0,14.0,26.0,0.768786,0.904762,0.836478,0.869281,0.05,0.3,159.0,147.0,144.0,147.0
8,21,265.0,7.0,27.0,0.886288,0.974265,0.907534,0.939716,0.05,0.3,292.0,272.0,266.0,272.0
9,22,255.0,11.0,28.0,0.867347,0.958647,0.901060,0.928962,0.05,0.3,283.0,266.0,257.0,262.0


In [27]:
import pandas as pd
import numpy as np

def gerar_tabela_classes_latex(df, caption="Métricas por classe", label="tab:classes"):
    # colunas que serão removidas
    remover = {
        "conf_thresh",
        "iou_thresh",
        "n_comp_label",
        "n_comp_pred",
        "true_pos",
        "false_pos",
        "false_neg",
    }
    
    df = df[[c for c in df.columns if c not in remover]].copy()

    # renomear colunas para nomes bonitos
    rename_map = {
        "class_name": "Classe",
        "avg_accuracy": "Accuracy",
        "avg_precision": "Precision",
        "avg_recall": "Recall",
        "avg_f1": "F1-Score",
        "n_img_label": "N Img. Label",
        "n_img_pred": "N Img. Pred",
    }
    df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})

    # formatação dos valores
    def fmt(x):
        if isinstance(x, (int, np.integer)):
            return str(x)
        if isinstance(x, (float, np.floating)):
            if float(x).is_integer():
                return str(int(x))
            return f"{x:.2f}"
        return str(x)

    df_fmt = df.apply(lambda col: col.map(fmt))

    # todas as colunas centralizadas
    col_format = " ".join(["c"] * len(df_fmt.columns))

    latex_body = df_fmt.to_latex(
        index=False,
        escape=False,
        header=True,
        longtable=False,
        column_format=col_format
    )

    linhas = latex_body.split("\n")
    linhas_limpo = [
        l for l in linhas
        if ("toprule" not in l and "midrule" not in l and "bottomrule" not in l)
    ]

    tabela = "\n".join(linhas_limpo)

    tabela = tabela.replace("\\\\", "\\\\ \\hline", 1)

    tabela_final = f"""
\\begin{{table}}[ht]
    \\centering
    \\caption{{{caption}}}
    \\label{{{label}}}

{tabela}

\\end{{table}}
""".strip()

    return tabela_final

In [28]:
print(gerar_tabela_classes_latex(resultados_dente, caption="Métricas por Classe", label="tab:sumario_dentes"))

\begin{table}[ht]
    \centering
    \caption{Métricas por Classe}
    \label{tab:sumario_dentes}

\begin{tabular}{c c c c c c c}
Classe & Accuracy & Precision & Recall & F1-Score & N Img. Label & N Img. Pred \\ \hline
11 & 0.87 & 0.97 & 0.89 & 0.93 & 264 & 268 \\
12 & 0.86 & 0.96 & 0.89 & 0.92 & 253 & 260 \\
13 & 0.86 & 0.97 & 0.88 & 0.93 & 264 & 265 \\
14 & 0.85 & 0.96 & 0.88 & 0.92 & 239 & 240 \\
15 & 0.84 & 0.95 & 0.88 & 0.91 & 231 & 236 \\
16 & 0.86 & 0.99 & 0.87 & 0.93 & 233 & 225 \\
17 & 0.84 & 0.96 & 0.87 & 0.91 & 239 & 238 \\
18 & 0.77 & 0.90 & 0.84 & 0.87 & 144 & 147 \\
21 & 0.89 & 0.97 & 0.91 & 0.94 & 266 & 272 \\
22 & 0.87 & 0.96 & 0.90 & 0.93 & 257 & 262 \\
23 & 0.86 & 0.96 & 0.90 & 0.93 & 257 & 263 \\
24 & 0.84 & 0.95 & 0.88 & 0.91 & 240 & 243 \\
25 & 0.84 & 0.95 & 0.89 & 0.92 & 239 & 244 \\
26 & 0.86 & 0.98 & 0.87 & 0.92 & 233 & 229 \\
27 & 0.85 & 0.97 & 0.87 & 0.92 & 248 & 246 \\
28 & 0.79 & 0.90 & 0.87 & 0.88 & 138 & 143 \\
31 & 0.89 & 0.97 & 0.91 & 0.94 & 264 & 269 \\

In [12]:
dados = pd.read_csv("/home/aninha/Desktop/Doutorado/Modelos/Métricas dos Modelos(Rascunho).csv", sep=";")
dados

,Achado,N,F1-Score,DMFT
0,Hígido,21018,"92,6",-
1,Restauração,8910,"75,7",F
2,Dente ausente,4746,"95,1",M
3,Condutos obturados,1384,"93,3",M
4,Dente incluso,717,"94,2",-
5,Prótese fixa sobre implante,178,"48,4",M
6,Dente impactado,351,"95,7",-
7,Dente semi-incluso,310,"91,2",-
8,Coroa unitária sobre dente,351,"88,1",F
9,Implante,262,"86,7",M


In [13]:
dados.columns

Index(['Achado', 'N', 'F1-Score', 'DMFT'], dtype='object')

In [14]:
import pandas as pd
import numpy as np

def gerar_tabela_achados_filtrados_latex(
    df,
    caption="Average F1-Score for selected findings",
    label="tab:selected_findings"
):
    df = df.copy()

    # Garante colunas esperadas
    cols_esperadas = ["Achado", "N", "F1-Score"]
    for c in cols_esperadas:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    # Achados a excluir (nomes exatamente como na tabela original)
    excluir = {
        "Hígido",
        "Dente semi-incluso",
        "Dente impactado",
        "Dente incluso",
        "Contenção",
        "Aparelho ortodôntico",
        "Placa",
        "Dente supranumerário",
        "Condutos obturados",
        "Retentor intrarradicular",
    }

    df = df[~df["Achado"].isin(excluir)].copy()

    # Converte F1-Score "92,6" -> 0.926 (decimal)
    df["F1-Score"] = (
        df["F1-Score"]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float) / 100.0
    )

    # Converte N para numérico
    df["N"] = pd.to_numeric(df["N"], errors="coerce")

    # Mapeia Achado remanescente para inglês
    achado_map = {
        "Restauração": "Restoration",
        "Dente ausente": "Missing tooth",
        "Prótese fixa sobre implante": "Fixed dental prosthesis on implant",
        "Coroa unitária sobre dente": "Single crown on tooth",
        "Implante": "Implant",
        "Raiz residual": "Residual root",
        "Coroa unitária sobre implante": "Single crown on implant",
        "Cárie": "Caries",
        "Prótese fixa sobre dente": "Fixed dental prosthesis",
        "Prótese protocolo": "Full-arch fixed prosthesis",
    }

    df["Finding"] = df["Achado"].map(achado_map)
    # Se algum não estiver no dicionário, mantém o original
    df["Finding"] = df["Finding"].fillna(df["Achado"])

    tabela = df[["Finding", "N", "F1-Score"]].copy()

    def fmt(x):
        if pd.isna(x):
            return "-"
        if isinstance(x, (int, np.integer)):
            return str(int(x))
        if isinstance(x, (float, np.floating)):
            return f"{x:.2f}"
        return str(x)

    tabela_fmt = tabela.apply(lambda col: col.map(fmt))

    col_format = "c c c"

    latex_body = tabela_fmt.to_latex(
        index=False,
        escape=False,
        header=True,
        longtable=False,
        column_format=col_format
    )

    linhas = latex_body.split("\n")
    linhas_limpo = [
        l for l in linhas
        if ("toprule" not in l and "midrule" not in l and "bottomrule" not in l)
    ]
    tabela_limpa = "\n".join(linhas_limpo)

    tabela_limpa = tabela_limpa.replace("\\\\", "\\\\ \\hline", 1)

    tabela_final = f"""
\\begin{{table}}[ht]
    \\centering
    \\caption{{{caption}}}
    \\label{{{label}}}

{tabela_limpa}

\\end{{table}}
""".strip()

    return tabela_final


In [15]:
latex = gerar_tabela_achados_filtrados_latex(
    dados,
    caption="Average F1-Score for caries, restorations, crowns, and prostheses",
    label="tab:f1_caries_restauracoes"
)

print(latex)

\begin{table}[ht]
    \centering
    \caption{Average F1-Score for caries, restorations, crowns, and prostheses}
    \label{tab:f1_caries_restauracoes}

\begin{tabular}{c c c}
Finding & N & F1-Score \\ \hline
Restoration & 8910 & 0.76 \\
Missing tooth & 4746 & 0.95 \\
Fixed dental prosthesis on implant & 178 & 0.48 \\
Single crown on tooth & 351 & 0.88 \\
Implant & 262 & 0.87 \\
Residual root & 280 & 0.60 \\
Single crown on implant & 149 & 0.82 \\
Caries & 654 & 0.34 \\
Fixed dental prosthesis & 42 & 0.69 \\
Full-arch fixed prosthesis & 5 & 0.57 \\
\end{tabular}


\end{table}


In [16]:
import pandas as pd
import numpy as np

def gerar_tabela_dmft_latex(
    df,
    caption="Average F1-Score by DMFT category",
    label="tab:dmft_summary"
):
    df = df.copy()

    # conferência de colunas
    cols_esperadas = ["Achado", "N", "F1-Score", "DMFT"]
    for c in cols_esperadas:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    # remove DMFT = "-" e Dente ausente
    df = df[(df["DMFT"] != "-") & (df["Achado"] != "Dente ausente")].copy()

    # converte F1-Score "92,6" -> 0.926 (decimal)
    df["F1-Score"] = (
        df["F1-Score"]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float) / 100.0
    )

    # N como inteiro
    df["N"] = pd.to_numeric(df["N"], errors="coerce").astype("Int64")

    # mapeamento Achado -> inglês (apenas os que sobram depois do filtro)
    achado_map = {
        "Restauração": "Restoration",
        "Condutos obturados": "Root-filled tooth",
        "Prótese fixa sobre implante": "Fixed dental prosthesis on implant",
        "Coroa unitária sobre dente": "Single crown on tooth",
        "Implante": "Implant",
        "Raiz residual": "Residual root",
        "Coroa unitária sobre implante": "Single crown on implant",
        "Cárie": "Caries",
        "Prótese fixa sobre dente": "Fixed dental prosthesis",
    }

    df["Finding"] = df["Achado"].map(achado_map).fillna(df["Achado"])

    # ordena por DMFT: D, M, F
    ordem_dmft = ["D", "M", "F"]
    df["DMFT"] = pd.Categorical(df["DMFT"], categories=ordem_dmft, ordered=True)
    df = df.sort_values(["DMFT", "Finding"]).reset_index(drop=True)

    # formatação dos valores
    def fmt_n(x):
        if pd.isna(x):
            return "-"
        return f"{int(x)}"

    def fmt_f1(x):
        if pd.isna(x):
            return "-"
        return f"{x:.2f}"

    # construção das linhas da tabela com multirow
    linhas = []
    linhas.append(r"\begin{tabular}{c c c c}")
    linhas.append(r"Finding & N & F1-Score & DMFT \\ \hline")

    for dmft_cat in ordem_dmft:
        df_g = df[df["DMFT"] == dmft_cat]
        if df_g.empty:
            continue

        k = len(df_g)
        first = True
        for _, row in df_g.iterrows():
            finding = row["Finding"]
            n_str = fmt_n(row["N"])
            f1_str = fmt_f1(row["F1-Score"])

            if first:
                dmft_cell = rf"\multirow{{{k}}}{{*}}{{{dmft_cat}}}"
                linha = rf"{finding} & {n_str} & {f1_str} & {dmft_cell} \\"
                first = False
            else:
                linha = rf"{finding} & {n_str} & {f1_str} & \\"

            linhas.append(linha)

        linhas.append(r"\hline")

    linhas.append(r"\end{tabular}")
    body = "\n".join(linhas)

    tabela_final = f"""
\\begin{{table}}[ht]
    \\centering
    \\caption{{{caption}}}
    \\label{{{label}}}

{body}

\\end{{table}}
""".strip()

    return tabela_final


In [18]:
latex = gerar_tabela_dmft_latex(
    dados,
    caption="Average F1-Score by DMFT category",
    label="tab:dmft_summary"
)
print(latex)

\begin{table}[ht]
    \centering
    \caption{Average F1-Score by DMFT category}
    \label{tab:dmft_summary}

\begin{tabular}{c c c c}
Finding & N & F1-Score & DMFT \\ \hline
Caries & 654 & 0.34 & \multirow{1}{*}{D} \\
\hline
Fixed dental prosthesis on implant & 178 & 0.48 & \multirow{5}{*}{M} \\
Implant & 262 & 0.87 & \\
Residual root & 280 & 0.60 & \\
Root-filled tooth & 1384 & 0.93 & \\
Single crown on implant & 149 & 0.82 & \\
\hline
Fixed dental prosthesis & 42 & 0.69 & \multirow{3}{*}{F} \\
Restoration & 8910 & 0.76 & \\
Single crown on tooth & 351 & 0.88 & \\
\hline
\end{tabular}

\end{table}


In [1]:
import pandas as pd

dados = pd.read_csv("/home/aninha/Desktop/Doutorado/Dados/anotadas/metrics_odontograma_limpo.csv")

dados.head()

,Unnamed: 0,dente,N,TP,FP,FN,TN,Accuracy,Precision,Recall,F1-score
0,0,11,963,846,0,3,114,0.996885,1.000000,0.996466,0.998230
1,1,12,963,838,10,0,115,0.989616,0.988208,1.000000,0.994069
2,2,13,963,873,2,11,77,0.986501,0.997714,0.987557,0.992609
3,3,14,963,715,17,10,221,0.971963,0.976776,0.986207,0.981469
4,4,15,963,698,7,19,239,0.973001,0.990071,0.973501,0.981716


In [3]:
# Columns to remove (absolute counts)
cols_remove = ["Unnamed: 0", "N", "TP", "FP", "FN", "TN"]

# Remove if they exist
dados = dados.drop(columns=[c for c in cols_remove if c in dados.columns])

# Rename to English
rename_dict = {
    "dente": "Tooth",
    "Accuracy": "Accuracy",
    "Precision": "Precision",
    "Recall": "Recall",
    "F1-score": "F1-Score"
}
dados = dados.rename(columns=rename_dict)

# Round numerical columns
metric_cols = ["Accuracy", "Precision", "Recall", "F1-Score"]
dados[metric_cols] = dados[metric_cols].apply(lambda x: x.round(2))

# Generate LaTeX table with hrule
latex_table = (
    "\\hrule\n"
    "\\begin{table}[H]\n"
    "\\centering\n"
    + dados.to_latex(index=False, float_format="%.2f")
    + "\\end{table}\n"
    "\\hrule\n"
)

print(latex_table)

\hrule
\begin{table}[H]
\centering
\begin{tabular}{rrrrr}
\toprule
Tooth & Accuracy & Precision & Recall & F1-Score \\
\midrule
11 & 1.00 & 1.00 & 1.00 & 1.00 \\
12 & 0.99 & 0.99 & 1.00 & 0.99 \\
13 & 0.99 & 1.00 & 0.99 & 0.99 \\
14 & 0.97 & 0.98 & 0.99 & 0.98 \\
15 & 0.97 & 0.99 & 0.97 & 0.98 \\
16 & 0.98 & 0.98 & 0.98 & 0.98 \\
17 & 0.98 & 0.99 & 0.99 & 0.99 \\
18 & 0.98 & 0.98 & 0.96 & 0.97 \\
21 & 0.99 & 1.00 & 0.99 & 1.00 \\
22 & 0.99 & 1.00 & 1.00 & 1.00 \\
23 & 0.98 & 1.00 & 0.99 & 0.99 \\
24 & 0.97 & 0.98 & 0.99 & 0.98 \\
25 & 0.97 & 1.00 & 0.96 & 0.98 \\
26 & 0.98 & 0.99 & 0.98 & 0.98 \\
27 & 0.97 & 0.98 & 0.98 & 0.98 \\
28 & 0.98 & 0.98 & 0.95 & 0.96 \\
31 & 0.99 & 1.00 & 0.99 & 1.00 \\
32 & 0.99 & 1.00 & 0.99 & 0.99 \\
33 & 0.99 & 1.00 & 0.99 & 1.00 \\
34 & 0.98 & 0.99 & 0.99 & 0.99 \\
35 & 0.98 & 1.00 & 0.98 & 0.99 \\
36 & 0.98 & 0.97 & 1.00 & 0.98 \\
37 & 0.97 & 0.98 & 0.98 & 0.98 \\
38 & 0.97 & 0.99 & 0.92 & 0.95 \\
41 & 0.99 & 1.00 & 0.99 & 0.99 \\
42 & 0.99 & 1.00 & 0.9

In [9]:
import pandas as pd

achados_ingles = {
    "aparelho ortodondico": "Orthodontic Appliance",
    "carie": "Caries",
    "condutos obturados": "Root Canal Treatment",
    "contencao": "Orthodontic Retainer",
    "coroa unitaria sobre dente": "Single Crown on Tooth",
    "coroa unitaria sobre implante": "Single Crown on Implant",
    "dente impactado": "Impacted Tooth",
    "dente incluso": "Embedded Tooth",
    "dente semi-incluso": "Semi-Embedded Tooth",
    "implante": "Dental Implant",
    "placa": "Dental Splint",
    "protese fixa sobre dente": "Fixed Prosthesis on Tooth",
    "protese fixa sobre implante": "Fixed Prosthesis on Implant",
    "protocolo": "Full-Arch Implant Prosthesis",
    "raiz residual": "Residual Root",
    "restauracao": "Restoration",
    "retentor intraradicular": "Intracanal Post",
}

def gerar_tabela_latex(df, caption="Metrics per finding", label="tab:metrics", colunas_excluir=None):

    df["Finding"] = df["Finding"].replace(achados_ingles)

    df = df.sort_values("Finding")

    # remove colunas indesejadas
    if colunas_excluir:
        df = df.drop(columns=[c for c in colunas_excluir if c in df.columns])
    
    # monta ambiente table
    latex = "\\begin{table}[H]\n"
    latex += "\\centering\n"
    latex += f"\\caption{{{caption}}}\n"
    latex += f"\\label{{{label}}}\n"

    # formatação das colunas
    col_format = "l" + "c" * (len(df.columns) - 1)
    latex += f"\\begin{{tabular}}{{{col_format}}}\n"

    # cabeçalho
    header = " & ".join(df.columns) + " \\\\"
    latex += header + "\n"

    # linhas da tabela
    for _, row in df.iterrows():
        line = " & ".join(
            f"{x:.2f}" if isinstance(x, (float, int)) else str(x)
            for x in row.values
        )
        latex += line + " \\\\\n"

    latex += "\\end{tabular}\n"
    latex += "\\end{table}\n"

    return latex


df_metrics = pd.read_csv("/home/aninha/Desktop/Doutorado/Dados/anotadas/anotacoes_12_2/metrics_auditoria_limpo.csv")

colunas_excluir = ["Unnamed: 0","N_images","True_events","Pred_events","TP_count","FP_count","FN_count","MAE_count","RMSE_count"]

# Gerar saída
latex_output = gerar_tabela_latex(df_metrics, colunas_excluir=colunas_excluir)
print(latex_output)


\begin{table}[H]
\centering
\caption{Metrics per finding}
\label{tab:metrics}
\begin{tabular}{lcccc}
Finding & Accuracy & Precision & Recall & F1-score \\
Caries & 0.05 & 1.00 & 0.05 & 0.10 \\
Dental Implant & 0.99 & 1.00 & 0.99 & 0.99 \\
Dental Splint & 0.89 & 1.00 & 0.89 & 0.94 \\
Embedded Tooth & 0.85 & 0.90 & 0.94 & 0.92 \\
Fixed Prosthesis on Implant & 0.91 & 0.91 & 1.00 & 0.95 \\
Fixed Prosthesis on Tooth & 0.89 & 0.92 & 0.96 & 0.94 \\
Full-Arch Implant Prosthesis & 1.00 & 1.00 & 1.00 & 1.00 \\
Impacted Tooth & 0.87 & 0.99 & 0.87 & 0.93 \\
Intracanal Post & 0.79 & 1.00 & 0.79 & 0.89 \\
Orthodontic Appliance & 0.93 & 0.98 & 0.96 & 0.97 \\
Orthodontic Retainer & 0.84 & 1.00 & 0.84 & 0.91 \\
Residual Root & 0.78 & 0.96 & 0.81 & 0.88 \\
Restoration & 0.77 & 0.95 & 0.80 & 0.87 \\
Root Canal Treatment & 0.97 & 1.00 & 0.97 & 0.98 \\
Semi-Embedded Tooth & 0.73 & 0.81 & 0.88 & 0.84 \\
Single Crown on Implant & 0.94 & 1.00 & 0.94 & 0.97 \\
Single Crown on Tooth & 0.50 & 0.99 & 0.50 & 0.66 